# Experiment: polyBERT Conductivity 4-Fold CV (Notebook)

Converted from `torch/polybert_con/train_polybert_conductivity_4fold.py`.

Objective:
- Train a polyBERT feature-based conductivity regressor with 4-fold cross validation.
- Preserve top-N high conductivity samples across folds.

Expected CSV columns:
- `SMILES`
- `CONDUCTIVITY` (float > 0)


## Usage Notes

- Update `CSV_PATH` in the config cell before running.
- First run may download `kuelumbus/polyBERT` from Hugging Face.
- This notebook uses feature extraction (no fine-tuning) + a sklearn regressor (`ridge` or `mlp`).


In [ ]:
# Optional (run once in this kernel)
# %pip install -U numpy pandas scikit-learn sentence-transformers tqdm

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from IPython.display import display
except Exception:
    display = print


def to_psmiles(s: str) -> str:
    if not isinstance(s, str):
        return ""
    tmp = s.replace("[*]", "__STAR__")
    tmp = tmp.replace("*", "[*]")
    tmp = tmp.replace("__STAR__", "[*]")
    return tmp


def make_folds(df: pd.DataFrame, k: int = 4, top_n: int = 10, seed: int = 42) -> np.ndarray:
    """Assign each row to a fold id in [0..k-1], ensuring top_n are distributed."""
    n = len(df)
    fold = np.full(n, -1, dtype=int)
    top_idx = df["CONDUCTIVITY"].nlargest(top_n).index.to_numpy()
    for i, idx in enumerate(top_idx):
        fold[idx] = i % k

    remaining = np.where(fold < 0)[0]
    rng = np.random.default_rng(seed)
    rng.shuffle(remaining)
    chunks = np.array_split(remaining, k)
    for i in range(k):
        fold[chunks[i]] = i
    return fold


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def compute_polybert_embeddings(texts: list[str], batch_size: int = 64, device: str | None = None) -> np.ndarray:
    """Return [N, 600] embeddings from kuelumbus/polyBERT."""
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("kuelumbus/polyBERT", device=device)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    return emb


def configure_plot_font_times_new_roman(verbose: bool = False):
    """Load Times New Roman from local Fonts folders and apply to matplotlib."""
    try:
        import matplotlib.pyplot as plt
        from matplotlib import font_manager as fm
    except Exception:
        return None, None

    cwd = Path.cwd().resolve()
    font_dirs = []
    for base in [cwd] + list(cwd.parents):
        for dn in ("Fonts", "fonts"):
            d = base / dn
            if d.is_dir() and d not in font_dirs:
                font_dirs.append(d)

    font_files = []
    for d in font_dirs:
        for pat in ("times*.ttf", "times*.otf", "times*.ttc", "Times*.ttf", "Times*.otf", "Times*.ttc"):
            font_files.extend(sorted(d.glob(pat)))

    chosen_name = "Times New Roman"
    chosen_path = None
    if font_files:
        # Register all styles (regular/bold/italic), then use regular if available.
        for fp in font_files:
            try:
                fm.fontManager.addfont(str(fp))
            except Exception:
                pass

        regular = None
        for fp in font_files:
            name = fp.name.lower()
            if name in {"times.ttf", "times new roman.ttf"}:
                regular = fp
                break
        chosen_path = regular if regular is not None else font_files[0]

        try:
            chosen_name = fm.FontProperties(fname=str(chosen_path)).get_name() or "Times New Roman"
        except Exception:
            chosen_name = "Times New Roman"

    plt.rcParams["font.family"] = chosen_name
    plt.rcParams["font.serif"] = [chosen_name, "Times New Roman", "DejaVu Serif"]
    plt.rcParams["axes.unicode_minus"] = False

    if verbose:
        print(f"Applied plot font: {chosen_name} | source={chosen_path}")
    return chosen_name, chosen_path




## Configuration

Set paths and model options here. The defaults mirror the original script.


In [ ]:
CSV_PATH = Path("simulation-trajectory-aggregate.csv")  # TODO: change if needed
OUTDIR = Path("torch/polybert_con/runs/polybert_cv_notebook")

KFOLD = 4
TOP_N = 10
SEED = 42
BATCH_SIZE = 64
DEVICE = 'cuda'               # e.g., "cuda" or "cpu"

REGRESSOR = "ridge"        # "ridge" or "mlp"
RIDGE_ALPHA = 1.0
MLP_HIDDEN = 64
MLP_MAX_ITER = 1000

CACHE_EMBEDDINGS = True
OVERWRITE_EXISTING_OUTPUTS = True  # if False, you can point OUTDIR to a new folder

if not CSV_PATH.exists():
    print(f"CSV not found yet: {CSV_PATH.resolve()}")
    print("Update CSV_PATH, then rerun this and following cells.")


In [ ]:
outdir = Path(OUTDIR)
outdir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
required_cols = {"SMILES", "CONDUCTIVITY"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV must contain columns: {sorted(required_cols)}; missing={sorted(missing)}")

df = df.copy()
df["PSMILES"] = df["SMILES"].map(to_psmiles)
df["log10_cond"] = np.log10(df["CONDUCTIVITY"].astype(float))

# Fold assignment with top-N balancing
df["fold"] = make_folds(df, k=KFOLD, top_n=TOP_N, seed=SEED)
df.to_csv(outdir / "fold_assignment.csv", index=False)

# Top-N distribution report
top_idx = df["CONDUCTIVITY"].nlargest(TOP_N).index.to_numpy()
top_counts = df.loc[top_idx, "fold"].value_counts().sort_index()
with open(outdir / "topN_distribution.txt", "w", encoding="utf-8") as f:
    f.write("Top-N fold distribution (by CONDUCTIVITY):\n")
    for fold_id in range(KFOLD):
        f.write(f"  fold {fold_id}: {int(top_counts.get(fold_id, 0))}\n")

print(f"Rows: {len(df)}")
print("Fold counts:")
print(df["fold"].value_counts().sort_index())
display(df.head())


In [ ]:
emb_path = outdir / "embeddings.npy"
if CACHE_EMBEDDINGS and emb_path.exists():
    X = np.load(emb_path)
    print(f"Loaded cached embeddings: {emb_path} | shape={X.shape}")
else:
    X = compute_polybert_embeddings(df["PSMILES"].tolist(), batch_size=BATCH_SIZE, device=DEVICE)
    if CACHE_EMBEDDINGS:
        np.save(emb_path, X)
        print(f"Saved embeddings: {emb_path} | shape={X.shape}")

y = df["log10_cond"].to_numpy().astype(np.float32)

if REGRESSOR == "ridge":
    reg = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
    model = Pipeline([("scaler", StandardScaler()), ("reg", reg)])
elif REGRESSOR == "mlp":
    reg = MLPRegressor(
        hidden_layer_sizes=(MLP_HIDDEN, MLP_HIDDEN // 2),
        activation="relu",
        random_state=SEED,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        n_iter_no_change=20,
    )
    model = Pipeline([("scaler", StandardScaler()), ("reg", reg)])
else:
    raise ValueError(f"Unsupported REGRESSOR={REGRESSOR}")

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


In [ ]:
rows = []
preds_all = np.full_like(y, np.nan, dtype=float)

for fold_id in tqdm(range(KFOLD)):
    tr = np.where(df["fold"].to_numpy() != fold_id)[0]
    va = np.where(df["fold"].to_numpy() == fold_id)[0]

    model.fit(X[tr], y[tr])
    pred = model.predict(X[va]).astype(float)
    preds_all[va] = pred

    mae = float(mean_absolute_error(y[va], pred))
    r = rmse(y[va], pred)
    rows.append(
        {
            "fold": fold_id,
            "n_train": int(len(tr)),
            "n_val": int(len(va)),
            "mae_log10": mae,
            "rmse_log10": r,
            "mae_factor_approx": float(10 ** mae),
            "rmse_factor_approx": float(10 ** r),
        }
    )

res = pd.DataFrame(rows)
res.to_csv(outdir / "cv_metrics.csv", index=False)

# Overall metrics (OOF)
oof_mae = float(mean_absolute_error(y, preds_all))
oof_rmse = rmse(y, preds_all)

summary_lines = [
    "polyBERT + regressor (4-fold OOF)",
    res.to_string(index=False),
    "",
    f"OOF MAE (log10):  {oof_mae:.6f}  (~x{10 ** oof_mae:.2f})",
    f"OOF RMSE (log10): {oof_rmse:.6f} (~x{10 ** oof_rmse:.2f})",
]
with open(outdir / "summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines) + "\n")

# Save per-sample predictions (Trajectory ID is optional)
base_cols = [c for c in ["Trajectory ID", "SMILES", "PSMILES", "CONDUCTIVITY", "log10_cond", "fold"] if c in df.columns]
out = df[base_cols].copy()
out["pred_log10_cond"] = preds_all
out.to_csv(outdir / "oof_predictions.csv", index=False)

display(res)
print(f"\nOOF MAE(log10)={oof_mae:.4f} (~x{10 ** oof_mae:.2f})")
print(f"OOF RMSE(log10)={oof_rmse:.4f} (~x{10 ** oof_rmse:.2f})")
display(out.head())
print(f"\nSaved outputs to: {outdir.resolve()}")


## Per-Band Metrics (Conductivity Quartile Split)

- Bands are built by splitting `CONDUCTIVITY` into four equal-count groups (quartiles).
- Metrics are computed per band using `log10_cond` MAE/RMSE.
- An additional `log10 MAE` for samples with `CONDUCTIVITY >= 1e-4` is also reported.
- You can switch to manual split mode by setting three thresholds in `MANUAL_THRESHOLDS`.


In [ ]:
# Per-band metrics by conductivity (quartile split)
BAND_SPLIT_MODE = "quartile"  # "quartile" or "manual"
MANUAL_THRESHOLDS = None      # e.g., (1e-6, 1e-5, 1e-4) on CONDUCTIVITY scale for (q1|q2|q3|q4)
COND_THRESHOLD_FOR_EXTRA_MAE = 1e-4  # additional subset metric on raw conductivity scale

eval_df = df.copy()
eval_df["pred_log10_cond"] = preds_all
eval_df = eval_df.dropna(subset=["pred_log10_cond"]).copy()


def assign_conductivity_bands(values: pd.Series, mode: str = "quartile", thresholds=None):
    values = values.astype(float)

    if mode == "manual":
        if thresholds is None or len(thresholds) != 3:
            raise ValueError("MANUAL_THRESHOLDS must be a tuple/list of three values: (q1_q2, q2_q3, q3_q4)")
        t1, t2, t3 = map(float, thresholds)
        if not (t1 < t2 < t3):
            raise ValueError(f"Thresholds must satisfy t1 < t2 < t3. Got: {t1}, {t2}, {t3}")
        bands = pd.cut(
            values,
            bins=[-np.inf, t1, t2, t3, np.inf],
            labels=["q1", "q2", "q3", "q4"],
            include_lowest=True,
        )
        info = {
            "mode": mode,
            "q1_q2_threshold": t1,
            "q2_q3_threshold": t2,
            "q3_q4_threshold": t3,
        }
        return bands.astype("string"), info

    if mode == "quartile":
        # Rank-based split guarantees 4 groups even when duplicated conductivity values exist.
        order = values.sort_values(kind="mergesort").index.to_numpy()
        labels = pd.Series(index=values.index, dtype="object")
        for label, idxs in zip(["q1", "q2", "q3", "q4"], np.array_split(order, 4)):
            labels.loc[idxs] = label
        q25, q50, q75 = values.quantile([0.25, 0.5, 0.75]).tolist()
        info = {
            "mode": mode,
            "approx_quantile_25": float(q25),
            "approx_quantile_50": float(q50),
            "approx_quantile_75": float(q75),
        }
        return labels.astype("string"), info

    raise ValueError(f"Unknown BAND_SPLIT_MODE={mode}")


eval_df["cond_band"], band_split_info = assign_conductivity_bands(
    eval_df["CONDUCTIVITY"],
    mode=BAND_SPLIT_MODE,
    thresholds=MANUAL_THRESHOLDS,
)

band_rows = []
for band in ["q1", "q2", "q3", "q4"]:
    g = eval_df[eval_df["cond_band"] == band].copy()
    if g.empty:
        continue

    mae = float(mean_absolute_error(g["log10_cond"], g["pred_log10_cond"]))
    r = rmse(g["log10_cond"], g["pred_log10_cond"])

    # Optional: same metrics on raw conductivity scale (can be dominated by high values)
    true_cond = g["CONDUCTIVITY"].astype(float).to_numpy()
    pred_cond = np.power(10.0, g["pred_log10_cond"].astype(float).to_numpy())
    mae_cond = float(mean_absolute_error(true_cond, pred_cond))
    rmse_cond = float(np.sqrt(mean_squared_error(true_cond, pred_cond)))

    band_rows.append(
        {
            "band": band,
            "n": int(len(g)),
            "cond_min": float(g["CONDUCTIVITY"].min()),
            "cond_median": float(g["CONDUCTIVITY"].median()),
            "cond_max": float(g["CONDUCTIVITY"].max()),
            "mae_log10": mae,
            "rmse_log10": r,
            "mae_factor_approx": float(10 ** mae),
            "rmse_factor_approx": float(10 ** r),
            "mae_cond": mae_cond,
            "rmse_cond": rmse_cond,
        }
    )

band_metrics = pd.DataFrame(band_rows)
band_metrics["band"] = pd.Categorical(band_metrics["band"], categories=["q1", "q2", "q3", "q4"], ordered=True)
band_metrics = band_metrics.sort_values("band").reset_index(drop=True)

band_metrics.to_csv(outdir / "cv_metrics_by_conductivity_band.csv", index=False)

print("Band split info:")
print(band_split_info)
display(band_metrics)

# Extra subset metric: conductivity >= threshold
mask_high_cond = eval_df["CONDUCTIVITY"].astype(float) >= float(COND_THRESHOLD_FOR_EXTRA_MAE)
subset_high_cond = eval_df.loc[mask_high_cond].copy()
if subset_high_cond.empty:
    print(f"No samples with CONDUCTIVITY >= {COND_THRESHOLD_FOR_EXTRA_MAE:g}")
else:
    true_cond_hi = subset_high_cond["CONDUCTIVITY"].astype(float).to_numpy()
    pred_cond_hi = np.power(10.0, subset_high_cond["pred_log10_cond"].astype(float).to_numpy())
    mae_cond_hi = float(mean_absolute_error(true_cond_hi, pred_cond_hi))
    rmse_cond_hi = float(np.sqrt(mean_squared_error(true_cond_hi, pred_cond_hi)))
    mae_log10_hi = float(mean_absolute_error(subset_high_cond["log10_cond"], subset_high_cond["pred_log10_cond"]))
    rmse_log10_hi = rmse(subset_high_cond["log10_cond"], subset_high_cond["pred_log10_cond"])

    cond_ge_metrics = pd.DataFrame([
        {
            "condition": f"CONDUCTIVITY >= {float(COND_THRESHOLD_FOR_EXTRA_MAE):g}",
            "n": int(len(subset_high_cond)),
            "mae_cond": mae_cond_hi,
            "rmse_cond": rmse_cond_hi,
            "mae_log10": mae_log10_hi,
            "rmse_log10": rmse_log10_hi,
            "mae_factor_approx": float(10 ** mae_log10_hi),
            "rmse_factor_approx": float(10 ** rmse_log10_hi),
        }
    ])
    cond_ge_metrics.to_csv(outdir / "cv_metrics_cond_ge_threshold.csv", index=False)
    print(f"log10 MAE for samples with CONDUCTIVITY >= {COND_THRESHOLD_FOR_EXTRA_MAE:g}: {mae_log10_hi:.6f} (~x{10 ** mae_log10_hi:.2f})")
    print(f"Raw conductivity MAE for samples with CONDUCTIVITY >= {COND_THRESHOLD_FOR_EXTRA_MAE:g}: {mae_cond_hi:.6e}")
    display(cond_ge_metrics)

# Optional inspection table
display(eval_df[["CONDUCTIVITY", "log10_cond", "pred_log10_cond", "fold", "cond_band"]].head())


## Novel SMILES Summary (`all_novel_smiles.csv`)

- Loads `torch/polybert_con/all_novel_smiles.csv` and reports mean conductivity plus top-10 samples.
- If no conductivity column exists (current file has only `smiles`), the notebook refits on full training data (`df`, `X`, `y`) and uses predicted conductivity.


In [ ]:
# Novel SMILES conductivity summary (actual if present, otherwise predicted)
NOVEL_CSV_PATH = "all_novel_smiles.csv"
NOVEL_TOP_K = 10
NOVEL_SMILES_COL_CANDIDATES = ["SMILES", "smiles"]
NOVEL_COND_COL_CANDIDATES = ["CONDUCTIVITY", "conductivity", "pred_CONDUCTIVITY", "pred_cond"]
SAVE_NOVEL_PRED_CSV = True
NOVEL_EXCLUDE_SUBSTRINGS = ["[*][*]"]  # ignore rows containing invalid/undesired repeated star markers


novel_df = pd.read_csv(NOVEL_CSV_PATH).copy()
print(f"Loaded novel CSV: {NOVEL_CSV_PATH} | shape={novel_df.shape}")
print(f"Columns: {list(novel_df.columns)}")

smiles_col = next((c for c in NOVEL_SMILES_COL_CANDIDATES if c in novel_df.columns), None)
if smiles_col is None:
    raise ValueError(f"No SMILES column found. Tried: {NOVEL_SMILES_COL_CANDIDATES}")

# Drop rows with undesired repeated star markers like "[*][*]"
before_n = len(novel_df)
mask_bad_smiles = pd.Series(False, index=novel_df.index)
for pat in NOVEL_EXCLUDE_SUBSTRINGS:
    mask_bad_smiles = mask_bad_smiles | novel_df[smiles_col].astype(str).str.contains(pat, regex=False, na=False)
removed_n = int(mask_bad_smiles.sum())
if removed_n > 0:
    novel_df = novel_df.loc[~mask_bad_smiles].copy()
    print(f"Filtered out {removed_n} rows matching {NOVEL_EXCLUDE_SUBSTRINGS}; remaining={len(novel_df)} / {before_n}")
if novel_df.empty:
    raise ValueError("No rows left after filtering invalid novel SMILES patterns.")

cond_col = next((c for c in NOVEL_COND_COL_CANDIDATES if c in novel_df.columns), None)

if cond_col is not None:
    # Actual conductivity exists in the file.
    novel_df["conductivity_for_summary"] = novel_df[cond_col].astype(float)
    source_label = f"actual ({cond_col})"
else:
    # No actual conductivity in the file (current all_novel_smiles.csv). Predict conductivity.
    print("No conductivity column in novel CSV. Predicting conductivity with a final model fit on all training data.")

    if "X" not in globals() or "y" not in globals() or "df" not in globals():
        raise RuntimeError("Run training/data cells first so that df, X, y are available.")

    # Rebuild the same regressor configuration and fit on all available training rows.
    if REGRESSOR == "ridge":
        final_reg = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
        final_model = Pipeline([("scaler", StandardScaler()), ("reg", final_reg)])
    elif REGRESSOR == "mlp":
        final_reg = MLPRegressor(
            hidden_layer_sizes=(MLP_HIDDEN, MLP_HIDDEN // 2),
            activation="relu",
            random_state=SEED,
            max_iter=MLP_MAX_ITER,
            early_stopping=True,
            n_iter_no_change=20,
        )
        final_model = Pipeline([("scaler", StandardScaler()), ("reg", final_reg)])
    else:
        raise ValueError(f"Unsupported REGRESSOR={REGRESSOR}")

    final_model.fit(X, y)

    novel_df["PSMILES"] = novel_df[smiles_col].map(to_psmiles)
    novel_emb = compute_polybert_embeddings(
        novel_df["PSMILES"].tolist(),
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )
    novel_df["pred_log10_cond"] = final_model.predict(novel_emb).astype(float)
    novel_df["pred_cond"] = np.power(10, novel_df["pred_log10_cond"])
    novel_df["conductivity_for_summary"] = np.power(10.0, novel_df["pred_log10_cond"].to_numpy())
    source_label = "predicted"

    if SAVE_NOVEL_PRED_CSV:
        pred_out = outdir / "all_novel_smiles_with_pred_conductivity.csv"
        novel_df.to_csv(pred_out, index=False)
        print(f"Saved predicted novel conductivities: {pred_out}")

mean_cond = float(novel_df["conductivity_for_summary"].astype(float).mean())
topk_df = (
    novel_df.sort_values("conductivity_for_summary", ascending=False)
    .head(NOVEL_TOP_K)
    .reset_index(drop=True)
)

print(f"Conductivity source: {source_label}")
print(f"Mean conductivity ({source_label}): {mean_cond:.6e}")
print(f"Top {NOVEL_TOP_K} samples by conductivity ({source_label}):")

cols_to_show = [c for c in [smiles_col, "conductivity_for_summary", "pred_log10_cond", "PSMILES"] if c in topk_df.columns]
display(topk_df[cols_to_show])

# Conductivity distribution plots
try:
    import matplotlib.pyplot as plt
    configure_plot_font_times_new_roman(verbose=False)
        
    cond_vals = pd.to_numeric(novel_df["conductivity_for_summary"], errors="coerce")
    cond_vals = cond_vals[np.isfinite(cond_vals)]
    cond_vals = cond_vals[cond_vals > 0]

    if len(cond_vals) == 0:
        print("No positive conductivity values available for plotting.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].hist(cond_vals, bins=50, color="#4C78A8", alpha=0.85)
        axes[0].set_xscale("log")
        axes[0].axvline(mean_cond, color="#E45756", linestyle="--", linewidth=1.5, label=f"mean={mean_cond:.2e}")
        axes[0].set_title(f"Conductivity Distribution ({source_label})")
        axes[0].set_xlabel("Conductivity")
        axes[0].set_ylabel("Count")
        axes[0].legend()

        log10_vals = np.log10(cond_vals.to_numpy())
        axes[1].hist(log10_vals, bins=50, color="#72B7B2", alpha=0.85)
        axes[1].axvline(np.log10(mean_cond), color="#E45756", linestyle="--", linewidth=1.5, label=f"mean(log10)={np.log10(mean_cond):.2f}")
        axes[1].set_title(f"log10 Conductivity Distribution ({source_label})")
        axes[1].set_xlabel("log10(Conductivity)")
        axes[1].set_ylabel("Count")
        axes[1].legend()

        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib is not installed. Run `%pip install matplotlib` to enable plots.")




## Compare Models: LOW / HIGH Distribution + Target Conductivity

- Recursively searches related CSV files under `torch/polybert_con/Decoder_Only`, `Encoder_Only`, `TransCVAE`, and `minGPT`.
- Condition labels are inferred from filenames/folders (`LOW`/`HIGH`) and overlaid per model.
- The x-axis uses conductivity in `log scale` by default.
- The y-axis uses `Count` (not density).
- LOW and HIGH are drawn on the same figure with split y-axes (`LOW`: left axis, `HIGH`: right axis).
- LOW uses the full y-scale from global LOW max count.
- HIGH uses `y_max = 2 * (model HIGH max count)`.
- Generation-condition and predicted-mean vertical lines are shown (except `minGPT`).
- Both flat model-folder CSV layout and nested `HIGH/LOW` subfolder layout are supported.
- Files without condition labels are treated as `ALL` (gray).


In [ ]:
# Compare generated conductivity distributions across models (LOW/HIGH)
MODEL_ROOT = Path(".")
MODEL_DIRS = {
    "TransCVAE": MODEL_ROOT / "TransCVAE",
    "LSTM_CVAE": MODEL_ROOT/"LSTM_CVAE",
    "Encoder_Only": MODEL_ROOT / "Encoder_Only",
    "TransCVAE_PSMILES": MODEL_ROOT / "TransCVAE_PSMILES",
    "Encoder_Only_PSMILES": MODEL_ROOT / "Encoder_Only_PSMILES",
    "LSTM_CVAE_PSMILES": MODEL_ROOT/"LSTM_CVAE_PSMILES",
    "minGPT": MODEL_ROOT / "minGPT",
}

CONDITION_ORDER = ["LOW", "HIGH"]
PLOT_CONDITION_ORDER = ["LOW", "HIGH", "ALL"]
CONDITION_COLORS = {
    "LOW": "#4C78A8",
    "HIGH": "#54A24B",
    "ALL": "#7F7F7F",
}

TARGET_COND_MODE = "dataset_tertile_median"  # "dataset_tertile_median" or "manual"
TARGET_COND_MAP_MANUAL = None  # e.g., {"LOW": 1e-6, "HIGH": 1e-2}

COMPARE_EXCLUDE_SUBSTRINGS = ["[*][*]"]
COMPARE_USE_EXISTING_PRED_COLUMNS = True
COMPARE_PLOT_BINS = 60
COMPARE_DENSITY = False
COMPARE_PLOT_X_SCALE = "log"  # "log" | "linear" | "log10"
COMPARE_MAX_ROWS_PER_FILE = None  # set int for debugging speed (e.g., 5000)
COMPARE_DROP_DUPLICATE_SMILES = True

COMPARE_FCD_TARGET_LINES = True
FCD_COND_INPUT_VALUES = {"LOW": -0.54, "HIGH": 20.0}  # from calculate_FCD_unified
FCD_PROPERTY_MODE = "log_minmax_zscore"  # "log_minmax_zscore" | "minmax"

if "df" not in globals() or "X" not in globals() or "y" not in globals():
    raise RuntimeError("Run the training/data/embedding cells first so that df, X, y are available.")

# Build the same final regressor on all training data (for conductivity prediction of generated samples)
if REGRESSOR == "ridge":
    final_reg = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
    final_model = Pipeline([("scaler", StandardScaler()), ("reg", final_reg)])
elif REGRESSOR == "mlp":
    final_reg = MLPRegressor(
        hidden_layer_sizes=(MLP_HIDDEN, MLP_HIDDEN // 2),
        activation="relu",
        random_state=SEED,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        n_iter_no_change=20,
    )
    final_model = Pipeline([("scaler", StandardScaler()), ("reg", final_reg)])
else:
    raise ValueError(f"Unsupported REGRESSOR={REGRESSOR}")

final_model.fit(X, y)

# One shared polyBERT model instance to avoid reloading per file
_polybert_holder = {"model": None}

def _encode_polybert(texts, batch_size=64, device=None):
    if _polybert_holder["model"] is None:
        from sentence_transformers import SentenceTransformer
        _polybert_holder["model"] = SentenceTransformer("kuelumbus/polyBERT", device=device)
    return _polybert_holder["model"].encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )

def _infer_cond_label_from_path(path: Path):
    text = "/".join([str(path).lower(), path.stem.lower()])
    if "high" in text:
        return "HIGH"
    if "low" in text:
        return "LOW"
    return "ALL"

def _pick_smiles_col(frame: pd.DataFrame):
    for c in ["SMILES", "smiles"]:
        if c in frame.columns:
            return c
    return None

def _predict_cond_from_smiles(smiles_series: pd.Series):
    psmiles = smiles_series.astype(str).map(to_psmiles)
    emb = _encode_polybert(psmiles.tolist(), batch_size=BATCH_SIZE, device=DEVICE)
    pred_log10 = final_model.predict(emb).astype(float)
    pred_cond = np.power(10.0, pred_log10)
    return pred_cond, pred_log10

def _extract_cond_values(frame: pd.DataFrame, smiles_col: str):
    if COMPARE_USE_EXISTING_PRED_COLUMNS:
        if "conductivity_for_summary" in frame.columns:
            vals = pd.to_numeric(frame["conductivity_for_summary"], errors="coerce")
            vals = vals[np.isfinite(vals)]
            vals = vals[vals > 0]
            if len(vals) > 0:
                return vals.to_numpy(dtype=float), None
        if "pred_log10_cond" in frame.columns:
            pred_log = pd.to_numeric(frame["pred_log10_cond"], errors="coerce")
            pred_log = pred_log[np.isfinite(pred_log)]
            if len(pred_log) > 0:
                return np.power(10.0, pred_log.to_numpy(dtype=float)), pred_log.to_numpy(dtype=float)
        for c in ["CONDUCTIVITY", "conductivity", "pred_CONDUCTIVITY", "pred_cond"]:
            if c in frame.columns:
                vals = pd.to_numeric(frame[c], errors="coerce")
                vals = vals[np.isfinite(vals)]
                vals = vals[vals > 0]
                if len(vals) > 0:
                    return vals.to_numpy(dtype=float), None

    pred_cond, pred_log = _predict_cond_from_smiles(frame[smiles_col])
    return pred_cond, pred_log

def _build_target_cond_map(train_df: pd.DataFrame):
    if TARGET_COND_MODE == "manual":
        if not TARGET_COND_MAP_MANUAL:
            raise ValueError("TARGET_COND_MAP_MANUAL must be set when TARGET_COND_MODE='manual'")
        return {k: float(TARGET_COND_MAP_MANUAL[k]) for k in CONDITION_ORDER}

    vals = pd.to_numeric(train_df["CONDUCTIVITY"], errors="coerce")
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]
    vals = vals.sort_values(kind="mergesort")

    target_map = {}
    for label, idxs in zip(CONDITION_ORDER, np.array_split(vals.index.to_numpy(), len(CONDITION_ORDER))):
        if len(idxs) == 0:
            target_map[label] = np.nan
        else:
            target_map[label] = float(pd.to_numeric(train_df.loc[idxs, "CONDUCTIVITY"], errors="coerce").median())
    return target_map

target_cond_map = _build_target_cond_map(df)
print("Target conductivity map (used for vertical lines):", target_cond_map)


def _invert_condition_to_raw(cond_input: float, conductivity_values: pd.Series, property_mode: str) -> float:
    vals = pd.to_numeric(conductivity_values, errors="coerce").to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]
    if vals.size == 0:
        return float("nan")

    if property_mode == "minmax":
        cmin = float(vals.min())
        cmax = float(vals.max())
        return float(cond_input * (cmax - cmin) + cmin)

    if property_mode == "log_minmax_zscore":
        log_vals = np.log1p(vals)
        lo = float(log_vals.min())
        hi = float(log_vals.max())
        span = hi - lo
        if span <= 0:
            return float(np.expm1(lo))
        mm = (log_vals - lo) / (span + 1e-12)
        mu = float(mm.mean())
        sd = float(mm.std() + 1e-8)
        mm_v = float(cond_input) * sd + mu
        log_v = mm_v * span + lo
        return float(np.expm1(log_v))

    raise ValueError(f"Unknown FCD_PROPERTY_MODE={property_mode}")

fcd_target_cond_map = {}
if COMPARE_FCD_TARGET_LINES and FCD_COND_INPUT_VALUES:
    for _k, _v in FCD_COND_INPUT_VALUES.items():
        try:
            fcd_target_cond_map[_k] = _invert_condition_to_raw(float(_v), df["CONDUCTIVITY"], FCD_PROPERTY_MODE)
        except Exception:
            fcd_target_cond_map[_k] = float("nan")
print("Generation-condition map (raw conductivity):", fcd_target_cond_map)

# Collect predicted conductivity distributions for each model/condition
model_cond_values = {m: {} for m in MODEL_DIRS}
rows = []

for model_name, model_dir in MODEL_DIRS.items():
    if not model_dir.exists():
        print(f"[WARN] Missing model dir: {model_dir}")
        continue

    candidate_paths = sorted(
        p for p in model_dir.rglob("*.csv")
        if p.is_file() and ("all_novel_smiles" in p.name.lower() or "novel" in p.name.lower())
    )
    if not candidate_paths:
        print(f"[WARN] No all_novel_smiles*.csv found under {model_dir}")
        continue

    grouped_paths = {c: [] for c in PLOT_CONDITION_ORDER}
    for p in candidate_paths:
        cond_label = _infer_cond_label_from_path(p)
        if cond_label in grouped_paths:
            grouped_paths[cond_label].append(p)

    print(f"\n[{model_name}] Found files:")
    for cond_label in PLOT_CONDITION_ORDER:
        print(f"  {cond_label}: {len(grouped_paths[cond_label])} file(s)")

    for cond_label in PLOT_CONDITION_ORDER:
        cond_paths = grouped_paths[cond_label]
        if not cond_paths:
            continue

        cond_frames = []
        total_removed = 0
        for p in cond_paths:
            frame = pd.read_csv(p).copy()
            smiles_col = _pick_smiles_col(frame)
            if smiles_col is None:
                print(f"  [SKIP] No smiles column in {p}")
                continue

            # Filter undesired patterns like [*][*]
            mask_bad = pd.Series(False, index=frame.index)
            for pat in COMPARE_EXCLUDE_SUBSTRINGS:
                mask_bad = mask_bad | frame[smiles_col].astype(str).str.contains(pat, regex=False, na=False)
            removed = int(mask_bad.sum())
            total_removed += removed
            if removed > 0:
                frame = frame.loc[~mask_bad].copy()

            if COMPARE_DROP_DUPLICATE_SMILES:
                before_dup = len(frame)
                frame = frame.drop_duplicates(subset=[smiles_col]).copy()
                removed_dup = before_dup - len(frame)
            else:
                removed_dup = 0

            if COMPARE_MAX_ROWS_PER_FILE is not None and len(frame) > int(COMPARE_MAX_ROWS_PER_FILE):
                frame = frame.sample(int(COMPARE_MAX_ROWS_PER_FILE), random_state=SEED).copy()

            frame["_source_file"] = str(p)
            frame["_smiles_col"] = smiles_col
            frame["_removed_dup"] = removed_dup
            cond_frames.append(frame)

        if not cond_frames:
            continue

        cond_df_all = pd.concat(cond_frames, ignore_index=True)
        smiles_col = cond_df_all["_smiles_col"].iloc[0]

        pred_cond_vals, pred_log10_vals = _extract_cond_values(cond_df_all, smiles_col=smiles_col)
        pred_cond_vals = np.asarray(pred_cond_vals, dtype=float)
        pred_cond_vals = pred_cond_vals[np.isfinite(pred_cond_vals)]
        pred_cond_vals = pred_cond_vals[pred_cond_vals > 0]

        if pred_cond_vals.size == 0:
            print(f"  [WARN] {model_name}/{cond_label}: no positive conductivity values after processing")
            continue

        model_cond_values[model_name][cond_label] = pred_cond_vals
        rows.append({
            "model": model_name,
            "condition": cond_label,
            "n_files": int(len(cond_paths)),
            "n_samples": int(pred_cond_vals.size),
            "mean_pred_cond": float(np.mean(pred_cond_vals)),
            "median_pred_cond": float(np.median(pred_cond_vals)),
            "target_cond": float(target_cond_map.get(cond_label, np.nan)),
            "target_over_mean": float(target_cond_map.get(cond_label, np.nan) / np.mean(pred_cond_vals)) if np.mean(pred_cond_vals) > 0 else np.nan,
            "min_pred_cond": float(np.min(pred_cond_vals)),
            "max_pred_cond": float(np.max(pred_cond_vals)),
            "removed_starstar_rows": int(total_removed),
        })

compare_summary_df = pd.DataFrame(rows)
if not compare_summary_df.empty:
    compare_summary_df["condition"] = pd.Categorical(compare_summary_df["condition"], categories=PLOT_CONDITION_ORDER, ordered=True)
    compare_summary_df = compare_summary_df.sort_values(["model", "condition"]).reset_index(drop=True)
display(compare_summary_df)

try:
    import matplotlib.pyplot as plt
    configure_plot_font_times_new_roman(verbose=False)
except ImportError:
    print("matplotlib is not installed. Run `%pip install matplotlib` to enable plots.")
    plt = None
if plt is not None:
    available_models = [m for m in MODEL_DIRS if model_cond_values.get(m)]
    if not available_models:
        print("No model-condition data available to plot.")
    else:
        all_vals = []
        for model_name in available_models:
            for cond_label in PLOT_CONDITION_ORDER:
                vals = model_cond_values.get(model_name, {}).get(cond_label)
                if vals is None or len(vals) == 0:
                    continue
                arr = np.asarray(vals, dtype=float)
                arr = arr[np.isfinite(arr)]
                arr = arr[arr > 0]
                if arr.size > 0:
                    all_vals.append(arr)

        if not all_vals:
            print("No plottable conductivity values.")
        else:
            global_vals = np.concatenate(all_vals)
            global_vals = global_vals[np.isfinite(global_vals)]
            global_vals = global_vals[global_vals > 0]

            if global_vals.size == 0:
                print("No finite positive conductivity values.")
            else:
                log_min = float(np.log10(global_vals.min()))
                log_max = float(np.log10(global_vals.max()))
                if not np.isfinite(log_min) or not np.isfinite(log_max):
                    log_min, log_max = -8.0, 2.0
                elif log_min == log_max:
                    log_min -= 0.5
                    log_max += 0.5

                if COMPARE_PLOT_X_SCALE == "log":
                    bin_edges = np.logspace(log_min, log_max, int(COMPARE_PLOT_BINS) + 1, base=10.0)
                elif COMPARE_PLOT_X_SCALE == "linear":
                    bin_edges = np.linspace(10.0 ** log_min, 10.0 ** log_max, int(COMPARE_PLOT_BINS) + 1)
                elif COMPARE_PLOT_X_SCALE == "log10":
                    bin_edges = np.linspace(log_min, log_max, int(COMPARE_PLOT_BINS) + 1)
                else:
                    raise ValueError(f"Unknown COMPARE_PLOT_X_SCALE={COMPARE_PLOT_X_SCALE}")

                # LOW full scale: global max count among LOW histograms.
                low_global_ymax = 0.0
                for model_name in available_models:
                    vals_low = model_cond_values.get(model_name, {}).get("LOW")
                    if vals_low is None or len(vals_low) == 0:
                        continue
                    arr_low = np.asarray(vals_low, dtype=float)
                    arr_low = arr_low[np.isfinite(arr_low)]
                    arr_low = arr_low[arr_low > 0]
                    if arr_low.size == 0:
                        continue
                    hist_input_low = np.log10(arr_low) if COMPARE_PLOT_X_SCALE == "log10" else arr_low
                    counts_low, _ = np.histogram(hist_input_low, bins=bin_edges, density=False)
                    if counts_low.size > 0:
                        low_global_ymax = max(low_global_ymax, float(np.nanmax(counts_low)))

                for model_name in available_models:
                    fig, ax_low = plt.subplots(1, 1, figsize=(11, 4))
                    ax_high = ax_low.twinx()

                    # LOW distribution on left y-axis
                    vals_low = model_cond_values.get(model_name, {}).get("LOW")
                    low_count_max = 0.0
                    if vals_low is not None and len(vals_low) > 0:
                        arr_low = np.asarray(vals_low, dtype=float)
                        arr_low = arr_low[np.isfinite(arr_low)]
                        arr_low = arr_low[arr_low > 0]
                        if arr_low.size > 0:
                            hist_input_low = np.log10(arr_low) if COMPARE_PLOT_X_SCALE == "log10" else arr_low
                            counts_low, _, _ = ax_low.hist(
                                hist_input_low,
                                bins=bin_edges,
                                density=False,
                                alpha=0.45,
                                color=CONDITION_COLORS.get("LOW", "#4C78A8"),
                                edgecolor=CONDITION_COLORS.get("LOW", "#4C78A8"),
                                label=f"LOW pred (n={len(arr_low)})",
                            )
                            if len(counts_low) > 0:
                                low_count_max = float(np.nanmax(counts_low))

                            if COMPARE_FCD_TARGET_LINES and model_name.lower() != "mingpt":
                                x_raw_low = fcd_target_cond_map.get("LOW", np.nan)
                                if np.isfinite(x_raw_low) and x_raw_low > 0:
                                    x_line_low = np.log10(x_raw_low) if COMPARE_PLOT_X_SCALE == "log10" else x_raw_low
                                    ax_low.axvline(
                                        x_line_low,
                                        color=CONDITION_COLORS.get("LOW", "#4C78A8"),
                                        linestyle="--",
                                        linewidth=1.5,
                                        alpha=0.9,
                                        label=f"Generation condition LOW={x_raw_low:.2e}",
                                    )

                                pred_mean_low = float(arr_low.mean())
                                mean_x_low = np.log10(pred_mean_low) if COMPARE_PLOT_X_SCALE == "log10" else pred_mean_low
                                ax_low.axvline(
                                    mean_x_low,
                                    color=CONDITION_COLORS.get("LOW", "#4C78A8"),
                                    linestyle=":",
                                    linewidth=1.8,
                                    alpha=0.9,
                                    label=f"Predicted mean LOW={pred_mean_low:.2e}",
                                )

                    # HIGH distribution on right y-axis
                    vals_high = model_cond_values.get(model_name, {}).get("HIGH")
                    high_count_max = 0.0
                    if vals_high is not None and len(vals_high) > 0:
                        arr_high = np.asarray(vals_high, dtype=float)
                        arr_high = arr_high[np.isfinite(arr_high)]
                        arr_high = arr_high[arr_high > 0]
                        if arr_high.size > 0:
                            hist_input_high = np.log10(arr_high) if COMPARE_PLOT_X_SCALE == "log10" else arr_high
                            counts_high, _, _ = ax_high.hist(
                                hist_input_high,
                                bins=bin_edges,
                                density=False,
                                alpha=0.35,
                                color=CONDITION_COLORS.get("HIGH", "#54A24B"),
                                edgecolor=CONDITION_COLORS.get("HIGH", "#54A24B"),
                                label=f"HIGH pred (n={len(arr_high)})",
                            )
                            if len(counts_high) > 0:
                                high_count_max = float(np.nanmax(counts_high))

                            if COMPARE_FCD_TARGET_LINES and model_name.lower() != "mingpt":
                                x_raw_high = fcd_target_cond_map.get("HIGH", np.nan)
                                if np.isfinite(x_raw_high) and x_raw_high > 0:
                                    x_line_high = np.log10(x_raw_high) if COMPARE_PLOT_X_SCALE == "log10" else x_raw_high
                                    ax_high.axvline(
                                        x_line_high,
                                        color=CONDITION_COLORS.get("HIGH", "#54A24B"),
                                        linestyle="--",
                                        linewidth=1.5,
                                        alpha=0.9,
                                        label=f"Generation condition HIGH={x_raw_high:.2e}",
                                    )

                                pred_mean_high = float(arr_high.mean())
                                mean_x_high = np.log10(pred_mean_high) if COMPARE_PLOT_X_SCALE == "log10" else pred_mean_high
                                ax_high.axvline(
                                    mean_x_high,
                                    color=CONDITION_COLORS.get("HIGH", "#54A24B"),
                                    linestyle=":",
                                    linewidth=1.8,
                                    alpha=0.9,
                                    label=f"Predicted mean HIGH={pred_mean_high:.2e}",
                                )

                    # x-axis scale / labels
                    if COMPARE_PLOT_X_SCALE == "log":
                        ax_low.set_xscale("log")
                        ax_high.set_xscale("log")
                        x_label = "predicted conductivity (log scale)"
                    elif COMPARE_PLOT_X_SCALE == "linear":
                        x_label = "predicted conductivity"
                    else:
                        x_label = "log10(predicted conductivity)"

                    ax_low.set_xlabel(x_label)
                    ax_low.set_ylabel("Count (LOW axis)", color=CONDITION_COLORS.get("LOW", "#4C78A8"))
                    ax_high.set_ylabel("Count (HIGH axis)", color=CONDITION_COLORS.get("HIGH", "#54A24B"))
                    ax_low.tick_params(axis='y', colors=CONDITION_COLORS.get("LOW", "#4C78A8"))
                    ax_high.tick_params(axis='y', colors=CONDITION_COLORS.get("HIGH", "#54A24B"))
                    ax_low.grid(alpha=0.2)

                    # y-limits rule
                    if np.isfinite(low_global_ymax) and low_global_ymax > 0:
                        ax_low.set_ylim(0, low_global_ymax)
                    elif low_count_max > 0:
                        ax_low.set_ylim(0, low_count_max)

                    high_ylim = high_count_max * 2.0
                    if np.isfinite(high_ylim) and high_ylim > 0:
                        ax_high.set_ylim(0, high_ylim)

                    # merged legend
                    h1, l1 = ax_low.get_legend_handles_labels()
                    h2, l2 = ax_high.get_legend_handles_labels()
                    merged = []
                    seen = set()
                    for h, l in list(zip(h1, l1)) + list(zip(h2, l2)):
                        if l in seen:
                            continue
                        seen.add(l)
                        merged.append((h, l))
                    if merged:
                        ax_low.legend([h for h, _ in merged], [l for _, l in merged], loc='best', fontsize=8)

                    ax_low.set_title(f"{model_name}: Predicted Conductivity (Count, Split Y-Axis)")
                    plt.tight_layout()
                    plt.show()



## Count Conductivity >= 1e-3 (By Model)

- Run this after the model-comparison cell (`Compare generated conductivity distributions...`).
- Counts samples satisfying `pred_cond >= 1e-3` for each model.


In [ ]:
# Count samples with predicted conductivity above a threshold (by model)
COUNT_THRESHOLD = 1e-3
SAVE_COUNT_TABLE = True

if "model_cond_values" not in globals():
    raise RuntimeError(
        "Run the comparison cell first so that `model_cond_values` is available."
    )

rows = []
rows_by_condition = []

for model_name in sorted(model_cond_values.keys()):
    cond_map = model_cond_values.get(model_name, {})
    arr_list = []

    for cond_label in sorted(cond_map.keys()):
        vals = np.asarray(cond_map[cond_label], dtype=float)
        vals = vals[np.isfinite(vals)]
        vals = vals[vals > 0]
        if vals.size == 0:
            continue

        n_ge = int((vals >= COUNT_THRESHOLD).sum())
        rows_by_condition.append(
            {
                "model": model_name,
                "condition": cond_label,
                "n_samples": int(vals.size),
                "n_ge_threshold": n_ge,
                "ratio_ge_threshold": float(n_ge / vals.size),
                "threshold": float(COUNT_THRESHOLD),
            }
        )
        arr_list.append(vals)

    if not arr_list:
        continue

    all_vals = np.concatenate(arr_list)
    n_ge_all = int((all_vals >= COUNT_THRESHOLD).sum())
    rows.append(
        {
            "model": model_name,
            "n_samples": int(all_vals.size),
            "n_ge_threshold": n_ge_all,
            "ratio_ge_threshold": float(n_ge_all / all_vals.size),
            "threshold": float(COUNT_THRESHOLD),
        }
    )

counts_by_model_df = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
counts_by_model_cond_df = pd.DataFrame(rows_by_condition).sort_values(["model", "condition"]).reset_index(drop=True)

print(f"Threshold: {COUNT_THRESHOLD:g}")
display(counts_by_model_df)
display(counts_by_model_cond_df)

if SAVE_COUNT_TABLE:
    out_model = outdir / f"counts_pred_cond_ge_{COUNT_THRESHOLD:g}_by_model.csv"
    out_cond = outdir / f"counts_pred_cond_ge_{COUNT_THRESHOLD:g}_by_model_condition.csv"
    counts_by_model_df.to_csv(out_model, index=False)
    counts_by_model_cond_df.to_csv(out_cond, index=False)
    print("Saved:", out_model)
    print("Saved:", out_cond)


## Reference Dataset Conductivity Distribution (True vs Predicted)

- The next cell compares true conductivity and OOF-predicted conductivity distributions on the reference dataset (`df`).
- Run the `Per-Band Metrics` cell first so that `eval_df` is available.


In [ ]:
# Reference dataset conductivity distribution: true vs predicted (OOF)
if "eval_df" not in globals():
    raise RuntimeError("Run the Per-Band Metrics cell first so that `eval_df` is available.")

ref_df = eval_df.copy()
ref_df["true_cond"] = pd.to_numeric(ref_df["CONDUCTIVITY"], errors="coerce")
ref_df["pred_cond"] = np.power(10.0, pd.to_numeric(ref_df["pred_log10_cond"], errors="coerce"))

plot_df = ref_df[["true_cond", "pred_cond"]].replace([np.inf, -np.inf], np.nan).dropna()
plot_df = plot_df[(plot_df["true_cond"] > 0) & (plot_df["pred_cond"] > 0)].copy()

if plot_df.empty:
    print("No positive conductivity values available for plotting.")
else:
    summary_df = pd.DataFrame([
        {
            "metric": "count",
            "true_cond": int(len(plot_df)),
            "pred_cond": int(len(plot_df)),
        },
        {
            "metric": "mean",
            "true_cond": float(plot_df["true_cond"].mean()),
            "pred_cond": float(plot_df["pred_cond"].mean()),
        },
        {
            "metric": "median",
            "true_cond": float(plot_df["true_cond"].median()),
            "pred_cond": float(plot_df["pred_cond"].median()),
        },
        {
            "metric": "min",
            "true_cond": float(plot_df["true_cond"].min()),
            "pred_cond": float(plot_df["pred_cond"].min()),
        },
        {
            "metric": "max",
            "true_cond": float(plot_df["true_cond"].max()),
            "pred_cond": float(plot_df["pred_cond"].max()),
        },
    ])
    display(summary_df)

    try:
        import matplotlib.pyplot as plt
        configure_plot_font_times_new_roman(verbose=False)
    except ImportError:
        print("matplotlib is not installed. Run `%pip install matplotlib` to enable plots.")
    else:
        true_vals = plot_df["true_cond"].to_numpy(dtype=float)
        pred_vals = plot_df["pred_cond"].to_numpy(dtype=float)

        true_log = np.log10(true_vals)
        pred_log = np.log10(pred_vals)
        x_min = float(min(true_log.min(), pred_log.min()))
        x_max = float(max(true_log.max(), pred_log.max()))
        if x_min == x_max:
            x_min -= 0.5
            x_max += 0.5
        bins_log = np.linspace(x_min, x_max, 61)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].hist(true_vals, bins=60, alpha=0.45, label="True CONDUCTIVITY", color="#4C78A8")
        axes[0].hist(pred_vals, bins=60, alpha=0.45, label="Pred CONDUCTIVITY (OOF)", color="#F58518")
        axes[0].set_xscale("log")
        axes[0].set_xlabel("Conductivity")
        axes[0].set_ylabel("Count")
        axes[0].set_title("Reference Dataset Conductivity (raw scale)")
        axes[0].legend()

        axes[1].hist(true_log, bins=bins_log, alpha=0.45, label="True log10(CONDUCTIVITY)", color="#4C78A8")
        axes[1].hist(pred_log, bins=bins_log, alpha=0.45, label="Pred log10(CONDUCTIVITY) (OOF)", color="#F58518")
        axes[1].set_xlabel("log10(Conductivity)")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Reference Dataset Conductivity (log10 scale)")
        axes[1].legend()

        plt.tight_layout()
        plt.show()





## Conductivity Eval Directly From `MODELS/FCD_runs`

- No file copy required.
- Reads `repeat_01~05/all_novel_smiles_condz_{low|high}.csv` directly.
- Computes per-repeat metrics and model-level `mean±std` summary.


In [ ]:
# Evaluate generated samples directly from MODELS/FCD_runs (repeat-wise)
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

def _resolve_fcd_runs_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd] + list(cwd.parents):
        cand = base / "MY_PAPER_RELATED" / "MODELS" / "FCD_runs"
        if cand.exists():
            return cand
        cand2 = base / "MODELS" / "FCD_runs"
        if cand2.exists():
            return cand2
    raise FileNotFoundError("Could not locate MODELS/FCD_runs from current working directory.")

FCD_RUNS_ROOT = _resolve_fcd_runs_root()
EVAL_MODELS = [
    "TransCVAE",
    "TransCVAE_PSMILES",
    "LSTM_CVAE",
    "LSTM_CVAE_PSMILES",
    "Encoder_Only",
    "Encoder_Only_PSMILES",
]
EVAL_COND_DIRS = {"LOW": "condz_low", "HIGH": "condz_high"}
EVAL_EXCLUDE_SUBSTRINGS = ["[*][*]"]
EVAL_DROP_DUPLICATE_SMILES = True
EVAL_SAVE = True
EVAL_OUT_DIR = FCD_RUNS_ROOT / "_conductivity_eval"

if "X" not in globals() or "y" not in globals() or "to_psmiles" not in globals():
    raise RuntimeError("Run the data + embedding cells first so that X, y, to_psmiles are available.")

# Fit one shared regressor on full reference set (same predictor for all generated samples)
if REGRESSOR == "ridge":
    final_reg_eval = Ridge(alpha=RIDGE_ALPHA, random_state=SEED)
    final_model_eval = Pipeline([("scaler", StandardScaler()), ("reg", final_reg_eval)])
elif REGRESSOR == "mlp":
    final_reg_eval = MLPRegressor(
        hidden_layer_sizes=(MLP_HIDDEN, MLP_HIDDEN // 2),
        activation="relu",
        random_state=SEED,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        n_iter_no_change=20,
    )
    final_model_eval = Pipeline([("scaler", StandardScaler()), ("reg", final_reg_eval)])
else:
    raise ValueError(f"Unsupported REGRESSOR={REGRESSOR}")

final_model_eval.fit(X, y)

_polybert_holder_eval = {"model": None}

def _encode_polybert_eval(texts, batch_size=64, device=None):
    if _polybert_holder_eval["model"] is None:
        from sentence_transformers import SentenceTransformer
        _polybert_holder_eval["model"] = SentenceTransformer("kuelumbus/polyBERT", device=device)
    return _polybert_holder_eval["model"].encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )


def _pick_smiles_col_eval(frame: pd.DataFrame):
    for c in ["SMILES", "smiles"]:
        if c in frame.columns:
            return c
    return None


def _predict_cond_from_smiles_eval(smiles_series: pd.Series):
    psmiles = smiles_series.astype(str).map(to_psmiles)
    emb = _encode_polybert_eval(psmiles.tolist(), batch_size=BATCH_SIZE, device=DEVICE)
    pred_log10 = final_model_eval.predict(emb).astype(float)
    pred_cond = np.power(10.0, pred_log10)
    return pred_cond, pred_log10


rows_repeat = []
model_cond_values = defaultdict(dict)

for model_name in EVAL_MODELS:
    model_dir = FCD_RUNS_ROOT / model_name
    if not model_dir.exists():
        print(f"[WARN] missing model dir: {model_dir}")
        continue

    for cond_label, cond_dir_name in EVAL_COND_DIRS.items():
        cond_dir = model_dir / cond_dir_name
        if not cond_dir.exists():
            print(f"[WARN] missing cond dir: {cond_dir}")
            continue

        repeat_dirs = sorted(p for p in cond_dir.glob("repeat_*") if p.is_dir())
        if not repeat_dirs:
            print(f"[WARN] no repeat_* dirs under: {cond_dir}")
            continue

        cond_values_all_repeats = []

        for repeat_dir in repeat_dirs:
            csv_path = repeat_dir / f"all_novel_smiles_{cond_dir_name}.csv"
            if not csv_path.exists():
                print(f"  [SKIP] missing {csv_path.name} in {repeat_dir.name}")
                continue

            frame = pd.read_csv(csv_path).copy()
            smiles_col = _pick_smiles_col_eval(frame)
            if smiles_col is None:
                print(f"  [SKIP] no smiles column in {csv_path}")
                continue

            # Filter invalid/undesired strings and deduplicate within repeat
            mask_bad = pd.Series(False, index=frame.index)
            for pat in EVAL_EXCLUDE_SUBSTRINGS:
                mask_bad = mask_bad | frame[smiles_col].astype(str).str.contains(pat, regex=False, na=False)
            frame = frame.loc[~mask_bad].copy()

            if EVAL_DROP_DUPLICATE_SMILES:
                frame = frame.drop_duplicates(subset=[smiles_col]).copy()

            if frame.empty:
                print(f"  [SKIP] empty after filtering: {csv_path}")
                continue

            pred_cond, _ = _predict_cond_from_smiles_eval(frame[smiles_col])
            pred_cond = np.asarray(pred_cond, dtype=float)
            pred_cond = pred_cond[np.isfinite(pred_cond) & (pred_cond > 0)]
            if pred_cond.size == 0:
                print(f"  [SKIP] no positive prediction values: {csv_path}")
                continue

            logv = np.log10(pred_cond)
            cond_values_all_repeats.append(pred_cond)

            rows_repeat.append({
                "model": model_name,
                "condition": cond_label,
                "repeat": repeat_dir.name,
                "source_csv": str(csv_path),
                "n_samples": int(pred_cond.size),
                "mean_log10_cond": float(np.mean(logv)),
                "median_log10_cond": float(np.median(logv)),
                "q90_log10_cond": float(np.quantile(logv, 0.9)),
                "hit_rate_ge_1e-4": float(np.mean(pred_cond >= 1e-4)),
                "hit_rate_ge_1e-3": float(np.mean(pred_cond >= 1e-3)),
            })

        if cond_values_all_repeats:
            model_cond_values[model_name][cond_label] = np.concatenate(cond_values_all_repeats)

conductivity_eval_by_repeat = pd.DataFrame(rows_repeat)
if conductivity_eval_by_repeat.empty:
    raise RuntimeError("No repeat-level conductivity evaluation rows were generated. Check FCD_RUNS_ROOT and CSV paths.")

conductivity_eval_summary = (
    conductivity_eval_by_repeat
    .groupby(["model", "condition"], as_index=False)
    .agg(
        repeats=("repeat", "nunique"),
        n_samples_mean=("n_samples", "mean"),
        mean_log10_cond_mean=("mean_log10_cond", "mean"),
        mean_log10_cond_std=("mean_log10_cond", "std"),
        median_log10_cond_mean=("median_log10_cond", "mean"),
        median_log10_cond_std=("median_log10_cond", "std"),
        q90_log10_cond_mean=("q90_log10_cond", "mean"),
        q90_log10_cond_std=("q90_log10_cond", "std"),
        hit_1e4_mean=("hit_rate_ge_1e-4", "mean"),
        hit_1e4_std=("hit_rate_ge_1e-4", "std"),
        hit_1e3_mean=("hit_rate_ge_1e-3", "mean"),
        hit_1e3_std=("hit_rate_ge_1e-3", "std"),
    )
    .sort_values(["condition", "model"]) 
    .reset_index(drop=True)
)

display(conductivity_eval_by_repeat)
display(conductivity_eval_summary)

if EVAL_SAVE:
    EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
    by_repeat_path = EVAL_OUT_DIR / "conductivity_eval_by_repeat.csv"
    summary_path = EVAL_OUT_DIR / "conductivity_eval_summary.csv"
    conductivity_eval_by_repeat.to_csv(by_repeat_path, index=False)
    conductivity_eval_summary.to_csv(summary_path, index=False)
    print(f"saved: {by_repeat_path}")
    print(f"saved: {summary_path}")

